# 03 — ML Model Training & Evaluation
## Nepal CPI Inflation Forecasting

**Models trained:**
- Ridge Regression (baseline)
- Random Forest
- XGBoost
- LightGBM
- Gradient Boosting
- Prophet (univariate time-series)
- ARIMA (statsmodels)

**Evaluation:** Walk-forward cross-validation + hold-out test set

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.preprocessing import (
    build_feature_matrix, train_test_split_ts,
    build_cpi_series
)
from src.models import (
    train_all, compare_models, walk_forward_cv,
    train_ridge, train_random_forest, train_xgboost,
    train_lightgbm, train_gradient_boosting,
    train_prophet, train_arima
)
from src.evaluate import (
    actual_vs_predicted_plot, feature_importance_plot, metrics_heatmap
)

print('Libraries loaded ✓')

In [ ]:
# ── Build feature matrix ───────────────────────────────────────────────────
df = build_feature_matrix()
print(f'Feature matrix shape: {df.shape}')
print(f'Date range: {df.index.min()} → {df.index.max()}')
print(f'\nFeatures:\n{list(df.columns)}')
df.tail()

In [ ]:
# ── Train/test split ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split_ts(df, test_ratio=0.2)
print(f'Train: {len(X_train)} samples  |  Test: {len(X_test)} samples')
print(f'Train period: {X_train.index.min()} → {X_train.index.max()}')
print(f'Test  period: {X_test.index.min()} → {X_test.index.max()}')

---
## Train All Models

In [ ]:
cpi_series = build_cpi_series()
results = train_all(X_train, X_test, y_train, y_test, cpi_series)
print('\nAll models trained ✓')

In [ ]:
# ── Model comparison table ─────────────────────────────────────────────────
comparison = compare_models(results)
print('=== Model Comparison ===')
display(comparison.set_index('Model').style
        .highlight_min(subset=['MAE','RMSE','MAPE'], color='#d5f5e3')
        .highlight_max(subset=['R2'], color='#d5f5e3')
        .format(precision=4))

In [ ]:
# ── Metrics heatmap ────────────────────────────────────────────────────────
fig = metrics_heatmap(comparison)
fig.show()

---
## Actual vs Predicted

In [ ]:
fig = actual_vs_predicted_plot(X_test.index, y_test.values, results)
fig.show()

---
## Feature Importance (Tree-based models)

In [ ]:
for model_key in ['random_forest', 'xgboost', 'lightgbm', 'gradient_boosting']:
    if model_key in results and 'feature_importance' in results[model_key]:
        fig = feature_importance_plot(
            results[model_key]['feature_importance'],
            model_name=results[model_key]['name']
        )
        fig.show()

---
## Walk-forward Cross-Validation

In [ ]:
X_all = df[[c for c in df.columns if c != 'cpi_yoy']]
y_all = df['cpi_yoy']

print('Walk-forward CV — XGBoost')
cv_results = walk_forward_cv(X_all, y_all, train_xgboost, n_splits=5, min_train=12)
cv_df = pd.DataFrame(cv_results)
display(cv_df)

print(f"\nMean RMSE: {cv_df['RMSE'].mean():.4f}  ±  {cv_df['RMSE'].std():.4f}")
print(f"Mean MAE : {cv_df['MAE'].mean():.4f}  ±  {cv_df['MAE'].std():.4f}")

---
## Prophet — 6-Month Ahead Forecast

In [ ]:
if 'prophet' in results and 'forecast' in results['prophet']:
    fc = results['prophet']['forecast']
    print('Prophet forecast (next 6 months):')
    display(fc[['ds','yhat','yhat_lower','yhat_upper']].tail(6).round(3))

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=cpi_series.index, y=cpi_series.values,
        name='Actual CPI', line=dict(color='black', width=2)
    ))
    fig.add_trace(go.Scatter(
        x=fc['ds'], y=fc['yhat'], name='Prophet Forecast',
        line=dict(color='#c0392b', width=2, dash='dot')
    ))
    if 'yhat_lower' in fc.columns:
        fig.add_trace(go.Scatter(
            x=pd.concat([fc['ds'], fc['ds'][::-1]]),
            y=pd.concat([fc['yhat_upper'], fc['yhat_lower'][::-1]]),
            fill='toself', fillcolor='rgba(192,57,43,0.15)',
            line=dict(color='rgba(0,0,0,0)'),
            name='95% CI'
        ))
    fig.update_layout(title='Prophet — Nepal CPI Inflation Forecast',
                       yaxis_title='CPI YoY (%)', template='plotly_white',
                       height=450)
    fig.show()

---
## ARIMA

In [ ]:
if 'arima' in results:
    arima_res = results['arima']
    print(arima_res.get('summary', ''))
    print('\nARIMA Metrics:', arima_res.get('metrics'))
    print('\n6-Month Forecast:')
    display(arima_res['forecast'].round(3))

---
## Residual Analysis (Best Model)

In [ ]:
# Pick best model by RMSE
best_key = comparison.set_index('Model')['RMSE'].idxmin() if not comparison.empty else None
if best_key:
    key_map = {
        'Ridge Regression': 'ridge',
        'Random Forest': 'random_forest',
        'XGBoost': 'xgboost',
        'LightGBM': 'lightgbm',
        'Gradient Boosting': 'gradient_boosting',
    }
    key = key_map.get(best_key, 'xgboost')
    if key in results and 'predictions' in results[key]:
        preds = results[key]['predictions']
        residuals = y_test.values - preds

        fig_res = px.scatter(x=preds, y=residuals,
                              title=f'Residual Plot — {best_key}',
                              labels={'x': 'Predicted', 'y': 'Residual'},
                              template='plotly_white')
        fig_res.add_hline(y=0, line_dash='dash', line_color='red')
        fig_res.show()

        # Distribution of residuals
        fig_hist = px.histogram(residuals, nbins=20,
                                 title=f'Residual Distribution — {best_key}',
                                 labels={'value': 'Residual'},
                                 template='plotly_white')
        fig_hist.show()